In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import warnings
import numpy as np
import rasterio
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

from gridr.chain.grid_resampling_chain import basic_grid_resampling_chain
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im
from chain_notebook_utils import (
    write_array, create_grid, create_star_polygon, plot_grid_on_image,
)

IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
if not IN_DOC_BUILD:
    from bokeh.io import output_notebook
    output_notebook()

# File paths shared across the chain tutorial series.
RASTER_IN          = "./grid_resampling_chain_raster_in.tif"
GRID_IN_F64        = "./grid_resampling_chain_grid_in_f64.tif"
output_raster_path = "./grid_resampling_chain_output_raster.tif"
output_mask_path   = "./grid_resampling_chain_output_mask.tif"

# Grid parameters shared across the chain tutorial series.
nrow, ncol = 50, 40
origin_pos  = np.array((0.3, 0.2))
origin_node = np.array((0., 0.))
v_row_y, v_row_x = 5.2, 1.2
v_col_y, v_col_x = -2.7, 7.1

grid_row_f64, grid_col_f64 = create_grid(
    nrow, ncol, origin_pos, origin_node,
    v_row_y, v_row_x, v_col_y, v_col_x, dtype=np.float64,
)

# Hybrid input creation: ensure the source raster and grid file exist on disk.
if not os.path.exists(RASTER_IN):
    write_array(mandrill, dtype=mandrill.dtype, fileout=RASTER_IN)
if not os.path.exists(GRID_IN_F64):
    write_array(np.array([grid_row_f64, grid_col_f64]),
                dtype=np.float64, fileout=GRID_IN_F64)

# Default output-dataset open arguments (resolution-dependent overrides
# are applied per-notebook below).
grid_resolution = (8, 8)
from gridr.core.grid.grid_commons import grid_full_resolution_shape
output_shape = grid_full_resolution_shape(
    shape=grid_row_f64.shape, resolution=grid_resolution,
)
raster_out_open_args = {
    "driver": "GTiff", "dtype": np.float64,
    "height": output_shape[0], "width": output_shape[1], "count": 1,
}
mask_out_open_args = {
    "driver": "GTiff", "dtype": np.uint8,
    "height": output_shape[0], "width": output_shape[1],
    "count": 1, "nbits": 1,
}

grid_mask_in_path        = "./grid_resampling_chain_grid_mask_in_u8_1_0.tif"
grid_mask_in_path_i8     = "./grid_resampling_chain_grid_mask_in_i8_0_m10.tif"
raster_mask_in_path_u8   = "./grid_resampling_chain_raster_mask_in_u8_1_0.tif"


# Grid mask

A *grid mask* flags individual grid nodes as invalid before resampling. Nodes marked invalid produce output samples set to `nodata_out`, and their validity is propagated to the output mask.

**What you'll learn**

- How to build and save a grid mask raster
- How to pass it through `grid_mask_in_ds`, `grid_mask_in_unmasked_value`
  and `grid_mask_in_band`
- How GridR supports alternative dtypes and validity conventions
  (`uint8` vs `int8`, valid value other than `1`)

## Setting things up

We reuse the source raster and grid file from the previous notebook. The grid mask itself is created below if not already present on disk.

## `uint8` grid mask (valid = 1, invalid = 0)

We build the mask in memory, marking as invalid any grid node whose source-frame coordinates fall inside a rectangular ROI, then write it to disk.

In [ ]:
grid_mask_in_valid_value, grid_mask_in_invalid_value = 1, 0

roi = np.array(((10, 40), (5, 100)))
grid_mask = np.full(grid_row_f64.shape, grid_mask_in_valid_value, dtype=np.uint8)
grid_mask[
    np.logical_and(
        np.logical_and(grid_row_f64 >= roi[0][0], grid_row_f64 <= roi[0][1]),
        np.logical_and(grid_col_f64 >= roi[1][0], grid_col_f64 <= roi[1][1]),
    )
] = grid_mask_in_invalid_value

write_array(grid_mask, dtype=np.uint8, fileout=grid_mask_in_path)

The figure below shows the grid nodes coloured by mask status:
- **red** = invalid (inside the ROI),
- **orange** = out of the source raster domain (also treated as invalid  during resampling),
- **blue** = valid.

In [ ]:
plot_grid_on_image(
    1.5, grid_row_f64, grid_col_f64, (10, 10),
    (mandrill.shape[1], mandrill.shape[2]),
    mask=grid_mask, win=None, raster_image=mandrill[0],
    prefix="grid_mask_input",
)

The mask is fed to the chain through three arguments:
- `grid_mask_in_ds` -- opened mask dataset,
- `grid_mask_in_unmasked_value` -- the value that means *valid*,
- `grid_mask_in_band` -- band index to read.

In [ ]:
with rasterio.open(GRID_IN_F64, "r") as grid_in_ds, \
        rasterio.open(RASTER_IN, "r") as array_src_ds, \
        rasterio.open(grid_mask_in_path, "r") as grid_mask_in_ds, \
        rasterio.open(output_raster_path, "w", **raster_out_open_args) as array_out_ds, \
        rasterio.open(output_mask_path, "w", **mask_out_open_args) as mask_out_ds:

    basic_grid_resampling_chain(
        grid_ds                     = grid_in_ds,
        grid_row_coords_band        = 1,
        grid_col_coords_band        = 2,
        grid_resolution             = grid_resolution,
        array_src_ds                = array_src_ds,
        array_src_bands             = 1,
        array_out_ds                = array_out_ds,
        interp                      = "cubic",
        nodata_out                  = 400,    # distinct value to make invalid areas visible
        mask_out_ds                 = mask_out_ds,
        grid_mask_in_ds             = grid_mask_in_ds,
        grid_mask_in_unmasked_value = grid_mask_in_valid_value,
        grid_mask_in_band           = 1,
    )

In [ ]:
with rasterio.open(output_raster_path, "r") as raster_ds, \
        rasterio.open(output_mask_path, "r") as mask_ds:
    plot_im({"output image": raster_ds.read(1), "output mask": mask_ds.read(1)},
            prefix="grid_mask_output")

## Alternative convention: `int8` (valid = 0, invalid = −10)

The chain supports any 8-bit integer dtype (`uint8` or `int8`) and any valid value. Only `grid_mask_in_unmasked_value` needs to match the chosen convention.

In [ ]:
grid_mask_in_valid_value_i8, grid_mask_in_invalid_value_i8 = 0, -10

grid_mask_i8 = np.full(grid_row_f64.shape, grid_mask_in_valid_value_i8, dtype=np.int8)
grid_mask_i8[
    np.logical_and(
        np.logical_and(grid_row_f64 >= roi[0][0], grid_row_f64 <= roi[0][1]),
        np.logical_and(grid_col_f64 >= roi[1][0], grid_col_f64 <= roi[1][1]),
    )
] = grid_mask_in_invalid_value_i8

write_array(grid_mask_i8, dtype=np.int8, fileout=grid_mask_in_path_i8)

with rasterio.open(GRID_IN_F64, "r") as grid_in_ds, \
        rasterio.open(RASTER_IN, "r") as array_src_ds, \
        rasterio.open(grid_mask_in_path_i8, "r") as grid_mask_in_ds, \
        rasterio.open(output_raster_path, "w", **raster_out_open_args) as array_out_ds, \
        rasterio.open(output_mask_path, "w", **mask_out_open_args) as mask_out_ds:

    basic_grid_resampling_chain(
        grid_ds                     = grid_in_ds,
        grid_row_coords_band        = 1,
        grid_col_coords_band        = 2,
        grid_resolution             = grid_resolution,
        array_src_ds                = array_src_ds,
        array_src_bands             = 1,
        array_out_ds                = array_out_ds,
        interp                      = "cubic",
        nodata_out                  = 0,
        mask_out_ds                 = mask_out_ds,
        grid_mask_in_ds             = grid_mask_in_ds,
        grid_mask_in_unmasked_value = grid_mask_in_valid_value_i8,
        grid_mask_in_band           = 1,
    )

In [ ]:
with rasterio.open(output_raster_path, "r") as raster_ds, \
        rasterio.open(output_mask_path, "r") as mask_ds:
    plot_im({"output image": raster_ds.read(1), "output mask": mask_ds.read(1)},
            prefix="grid_mask_out_int8")